

# Final Project

#### Colin Frishberg (cpfrish@berkeley.edu), Terra Jiang (yjiang66@berkeley.edu), Sai Sriya Mudigonda (smudigonda@berkeley.edu), Rahil Sharma (rahilsharma@berkeley.edu)


## Data Processing

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
import altair as alt
import seaborn as sns

alt.data_transformers.enable("vegafusion")

In [ ]:
%pip install "vegafusion[embed]>=1.5.0"

In [ ]:
# Install vl-convert-python if not already installed (works in both plain Python and Jupyter)
try:
    import vl_convert
except Exception:
    import sys, subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "vl-convert-python>=1.6.0"]
    )
    import vl_convert

In [ ]:
# from google.colab import files
# uploaded = files.upload()

In [ ]:
df = pd.read_csv("diabetic_data.csv")
raw_df = pd.read_csv("diabetic_data.csv")
display(df.head())

### Key for Drug descriptions ##

Up: The dosage of the drug was increased for the patient during their encounter.

Down: The dosage was decreased.

Steady: The patient is on the drug, and the dosage was not changed.

No: The patient was not prescribed this specific drug.

In [ ]:
print(f"There are {len(df.columns)} columns:\n", df.columns)

#### Drop duplicates and unnecessary columns

In [ ]:
df = df.drop_duplicates(subset=["patient_nbr"], keep="first")

In [ ]:
# checking to see how many missing values there are
(df == "?").sum()

# columns to keep based on descriptions / number of missing values
# race (split into indicator variables)
# gender (split into indicator variables)
# age (split into indicator variables)
# time_in_hospital (int)
# num_lab_procedures (int)
# num_procedures (int)
# num_medications (int)
# number_outpatient (int)
# number_emergency (int)
# number_inpatient (int)
# number_diagnoses (int)
# keeping all the indicator variables for medication / medication changes
# target - multiclass classification

# dropping the unnecessary columns
df = df.drop(
    columns=[
        "encounter_id",
        "patient_nbr",
        "weight",
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
        "payer_code",
        "medical_specialty",
        "diag_1",
        "diag_2",
        "diag_3",
        "max_glu_serum",
        "A1Cresult",
    ],
    axis=1,
)

#### Collapse age and race

In [ ]:
# Convert age buckets like "[70-80)" to numeric midpoint (75)
def age_to_mid(age_bucket):
    if pd.isna(age_bucket):
        return np.nan
    try:
        lo, hi = age_bucket.strip("[]").split("-")
        lo = int(lo)
        hi = int(hi.strip(")"))
        return (lo + hi) / 2
    except Exception:
        return np.nan


df["age_mid"] = df["age"].apply(age_to_mid)
df.drop(columns=["age"], inplace=True)

In [ ]:
# Collapse low count race categories into 'Other'
if "race" in df.columns:
    df["race"] = df["race"].fillna("Unknown")
    top = df["race"].value_counts().nlargest(4).index
    df["race_collapsed"] = df["race"].where(df["race"].isin(top), other="Other")
    df.drop(columns=["race"], inplace=True)

#### Categorical Features

In [ ]:
# creating indicator variables for the categorical features (one-hot encoding)
df = pd.get_dummies(df, columns=["race_collapsed", "gender"], dtype=int)

# Encode ordinal values according to the prescription status of the visit
prescription_map = {
    "No": 0,  # The drug was not prescribed
    "Down": 1,  # The dosage was decreased
    "Steady": 2,  # The dosage did not change
    "Up": 3,
}  # The dosage was increased during the encounter
cat_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
]
for cat_col in cat_columns:
    df[cat_col] = df[cat_col].map(prescription_map)

#### Binary features and Outcome of Interest

In [ ]:
# displaying the data
print("The number of columns are: ", len(df.columns))
display(df.head())
print(df.columns)

# recoding change and diabetesMed to 0 for No and 1 to Yes
df["change"] = np.where(df["change"] == "No", 0, 1)
df["diabetesMed"] = np.where(df["diabetesMed"] == "No", 0, 1)

# mapping the target into 2 classes
readmission_map = {
    "NO": 0,  # no readmission recorded
    ">30": 1,  # readmitted (over 30 days)
    "<30": 1,  # readmitted (within 30 days)
}

df["readmitted"] = df["readmitted"].map(readmission_map)

display(df.head())

df["readmitted"].value_counts()


In [ ]:
df.describe()

#### Shuffling and splitting

In [ ]:
# randomly shuffling the data
display(df.head())

indices = np.arange(len(df))

shuffled_indices = np.random.permutation(indices)

df = df.iloc[shuffled_indices].reset_index(drop=True)

display(df.head())

In [ ]:
# splitting X and Y data
X = df.copy().drop(columns=["readmitted"], axis=1)
Y = df.copy()["readmitted"]

In [ ]:
# splitting the data into train, val, test (60/20/20)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, train_size=0.8, random_state=1234
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train, Y_train, train_size=0.75, random_state=1234
)

#### Continuous feature standardizations

In [ ]:
# standardizing the continous features between 0 and 1
columns_to_standardize = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
]

scaler = MinMaxScaler()

X_train[columns_to_standardize] = scaler.fit_transform(X_train[columns_to_standardize])
X_val[columns_to_standardize] = scaler.transform(X_val[columns_to_standardize])
X_test[columns_to_standardize] = scaler.transform(X_test[columns_to_standardize])

print(f"The shape of X_train is {X_train.shape}")
print(f"\nThe shape of Y_train is {Y_train.shape}")
print(f"\nThe shape of X_val is {X_val.shape}")
print(f"\nThe shape of Y_val is {Y_val.shape}")
print(f"\nThe shape of X_test is {X_test.shape}")
print(f"\nThe shape of Y_test is {Y_test.shape}")


In [ ]:
display(X_train.head())
display(Y_train.head())

## EDA

In [ ]:
print(f"Number of columns in X_train: {len(X_train.columns)}")
print(f"List of columns in X_train: {X_train.columns.tolist()}")


In [ ]:
# combine features and target from training set
# visualize how features relate to target variable
train_combined = X_train.copy()
train_combined["readmitted"] = Y_train

train_combined["readmitted"].value_counts(normalize=True)

#### Continuous features

In [ ]:
# distribution of continuous features
import matplotlib.pyplot as plt
import math
import seaborn as sns

continuous_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
]

num_features = len(continuous_features)
num_cols = 2
num_rows = 5

fig, axes = plt.subplots(nrows=num_rows, ncols=num_cols, figsize=(12, 4 * num_rows))
for i, col in enumerate(continuous_features):
    row = i // num_cols
    col_index = i % num_cols
    ax = axes[row, col_index]

    sns.histplot(train_combined[col], kde=True, bins=30, ax=ax)
    ax.set_title(f"Distribution of {col}", fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel("Count of Visits")

plt.tight_layout()
plt.show()


In [ ]:
# --- Altair Recode + Box Plot functionality) ---

continuous_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "age_mid",
]


def plot_distributions_altair_with_kde(df: pd.DataFrame, numeric_cols):
    """
    Generates histograms with a KDE overlay and box plots side-by-side.
    """
    charts = []
    for col in numeric_cols:
        is_discrete_int = (
            pd.api.types.is_integer_dtype(df[col]) and df[col].nunique() < 30
        )
        x_binning = alt.Bin(step=1) if is_discrete_int else alt.Bin(maxbins=30)

        base = alt.Chart(df).properties(height=200)

        hist = (
            base.mark_bar(opacity=0.6)
            .encode(
                alt.X(
                    f"{col}:Q",
                    bin=x_binning,
                    title=col,
                    axis=alt.Axis(format="d", labelAngle=0),
                ),
                alt.Y("count():Q", title="Count", axis=alt.Axis(titleColor="#1f77b4")),
            )
            .properties(width=350)
        )

        density = (
            base.transform_density(col, as_=[col, "density"])
            .mark_line(color="orange", strokeWidth=3)
            .encode(
                x=f"{col}:Q",
                y=alt.Y(
                    "density:Q", axis=alt.Axis(title="Density", titleColor="orange")
                ),
            )
        )

        distribution_plot = (
            alt.layer(hist, density)
            .resolve_scale(y="independent")
            .properties(title=f"Distribution of {col}")
        )

        boxplot = (
            alt.Chart(df)
            .mark_boxplot()
            .encode(
                alt.X("readmitted:N", title="Readmitted"), alt.Y(f"{col}:Q", title=col)
            )
            .properties(title=f"{col} by Readmission", width=200, height=200)
        )

        combined_chart = distribution_plot | boxplot
        charts.append(combined_chart)

    final_chart = alt.vconcat(*charts)

    return final_chart


final_chart = plot_distributions_altair_with_kde(df, continuous_features)
final_chart.display()

#### Categorical features

In [ ]:
# distribution of key categorical features

categorical_features = {
    "race": [col for col in train_combined.columns if col.startswith("race_")],
    "gender": [col for col in train_combined.columns if col.startswith("gender_")],
    #'age': [col for col in train_combined.columns if col.startswith('age_')]
}

for categ_name, cols in categorical_features.items():
    plt.figure(figsize=(8, 4))
    counts = train_combined[cols].sum().sort_values(ascending=False)
    sns.barplot(x=counts.index, y=counts.values)
    plt.title(f"Distribution of {categ_name}")
    # plt.xlabel('Category')
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

#### Binary features

In [ ]:
# distribution of binary features
binary_features = ["change", "diabetesMed"]

for col in binary_features:
    plt.figure(
        figsize=(
            4,
            4,
        )
    )
    sns.countplot(data=train_combined, x=col)
    plt.title(f"Distribution of {col}")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

#### Heatmap

In [ ]:
# correlation between continuous features

plt.figure(figsize=(8, 6))
sns.heatmap(train_combined[continuous_features].corr(), annot=True, fmt=".2f")
plt.title("Correlation Matrix of Continuous Features")

#### Mutual Information Scores

In [ ]:
# correlation between features and target
# point-biseral correlation

X = train_combined.drop(columns=["readmitted"])
y = train_combined["readmitted"]

mutual_info_scores = mutual_info_classif(X, y, random_state=1234)
mutual_info_df = pd.DataFrame(
    {"Feature": X.columns, "Mutual Info Score": mutual_info_scores}
)
mutual_info_df = mutual_info_df.sort_values(by="Mutual Info Score", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=mutual_info_df.head(15), x="Mutual Info Score", y="Feature")
plt.title("Top Features by Mutual Information with Readmission")
plt.xlabel("Mutual Information Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## Visualizations ##

Visualization requirements: Include multiple detailed plots that effectively communicate your data insights: ensure all plots have properly labeled x and y axes; include descriptive titles; add legends where appropriate; consider using multiple plot types (histograms, scatter plots, box plots, heatmaps, etc.) to highlight different aspects of your data; accompany each visualization with interpretations of what the patterns reveal.

In [ ]:
viz_df = df.copy()

In [ ]:
viz_df.head()

#### Kernel Density Estimates

In [ ]:
# Kernel Density Estimates for continuous features
from vega_datasets import data

kde_vars = continuous_features


def create_kde_plots(df, kde_vars, ncols=2):
    """
    Generates and combines KDE plots into a grid for given features

    Args:
        df (pd.Dataframe)
        kde_vars(list): List of columns to plot
        ncols(int): Number of columns in the grid

    Returns:
        alt.vconcat: Grid of KDE plots
    """
    charts = []
    for var in kde_vars:
        chart = (
            alt.Chart(df)
            .transform_density(
                density=var,
                as_=[var, "Density"],  # The output fields for the value and its density
                groupby=["readmitted"],
            )
            .mark_area(opacity=0.5)
            .encode(
                x=alt.X(f"{var}:Q", title=var.replace("_", " ").title()),
                y=alt.Y("Density:Q"),
                # Color by the 'readmitted' class
                color=alt.Color("readmitted:N", title="Readmitted Class"),
            )
            .properties(
                title=f"Distribution of {var.replace('_', ' ').title()}",
                width=300,
                height=200,
            )
        )
        charts.append(chart)

        # Grid creations
        rows = [
            alt.hconcat(*charts[i : i + ncols]) for i in range(0, len(charts), ncols)
        ]
    # Combine all created charts vertically and return them
    return alt.vconcat(*rows)


In [ ]:
kde_plots = create_kde_plots(viz_df, kde_vars, ncols=4)
kde_plots


#### Box Plots

In [ ]:
# Box Plots for distributions of key predictors to outcome

box_plots = []
for feature in continuous_features:
    # Create box plot for each feature against readmission
    chart = (
        alt.Chart(viz_df)
        .mark_boxplot()
        .encode(
            x=alt.X("readmitted:N", title="Readmitted Class (0: NO, 1: >30, 2: <30)"),
            y=alt.Y(f"{feature}:Q", title=feature.replace("_", " ").title()),
            color=alt.Color("readmitted:N", title="Readmitted Class").scale(
                scheme="category20"
            ),
            tooltip=["readmitted", feature],
        )
        .properties(
            title=f"Box Plot of {feature.replace('_', ' ').title()} by Readmission Status",
            width=300,
            height=200,
        )
    )
    box_plots.append(chart)

# Arrange the box plots in a grid
ncols = 3
rows = [alt.hconcat(*box_plots[i : i + ncols]) for i in range(0, len(box_plots), ncols)]
alt.vconcat(*rows)

#### Violin Plots

In [ ]:
# Violin Plots
continuous_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

violin_plots = []
for feature in continuous_features:
    chart = (
        alt.Chart(raw_df)
        .transform_density(
            density=feature, as_=[feature, "density"], groupby=["readmitted"]
        )
        .mark_area(orient="horizontal")
        .encode(
            y=alt.Y(f"{feature}:Q", title=feature.replace("_", " ").title()),
            x=alt.X(
                "density:Q",
                stack="center",
                impute=None,
                title=None,
                axis=alt.Axis(labels=False, values=[0], grid=False, ticks=True),
            ),
            color=alt.Color(
                "readmitted:N", legend=alt.Legend(title="Readmitted")
            ).scale(scheme="tableau20"),
        )
        .properties(
            title=f"Distribution of {feature.replace('_', ' ').title()} by Readmission",
            width=300,
            height=250,
        )
    )
    violin_plots.append(chart)

# Arrange the violin plots in a grid
ncols = 4
rows_violin = [
    alt.hconcat(*violin_plots[i : i + ncols])
    for i in range(0, len(violin_plots), ncols)
]
final_violin_chart = alt.vconcat(*rows_violin)


final_violin_chart

In [ ]:
# Build a transformer model for readmission prediction
def build_transformer_model(input_shape, num_classes):
    inputs = keras.Input(shape=(input_shape,))

    # Embedding layer
    x = keras.layers.Dense(192, activation="relu")(inputs)
    x = keras.layers.Reshape((1, 192))(x)

    # Transformer block
    attention_output = keras.layers.MultiHeadAttention(num_heads=8, key_dim=24)(x, x)
    x = keras.layers.Add()([x, attention_output])
    x = keras.layers.LayerNormalization()(x)

    ffn_output = keras.layers.Dense(2048, activation="relu")(x)
    ffn_output = keras.layers.Dense(192)(ffn_output)
    x = keras.layers.Add()([x, ffn_output])
    x = keras.layers.LayerNormalization()(x)

    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(64, activation="relu")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
# Compile and train the model
input_shape = X_train.shape[1]
num_classes = 2  # Binary classification (0: No readmission, 1: Readmission)

model_transformer = build_transformer_model(input_shape, num_classes)
# Using sparse_categorical_crossentropy because the model outputs 2 logits (softmax)
model_transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model_transformer.fit(
    X_train,
    Y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_val, Y_val),
)

## Methodology

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import keras_tuner as kt
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Find the value counts and percentages for each output class
print("Value counts:")
display(Y_train.value_counts())
print("Value counts (%):")
display(Y_train.value_counts() / Y_train.shape[0] * 100)

### Model 1 (baseline model) - Logistic Regression

In [ ]:
# Use the features that have high scores from the mutual information scores plot
high_info_score = list(set(list(mutual_info_df["Feature"][:5]) + ["time_in_hospital"]))
print(
    f"Bottom 5 features from the mutual information scores plot (low score features are more independent from other features)\n\t{high_info_score}"
)
X_train_m1 = X_train[high_info_score]
X_val_m1 = X_val[high_info_score]
X_test_m1 = X_test[high_info_score]

# Make copies for model 1 Y datasets
Y_train_m1 = Y_train.copy()
Y_val_m1 = Y_val.copy()
Y_test_m1 = Y_test.copy()

# Reprint the shapes for each dataset
print("\nShape of X_train: ", X_train_m1.shape)
print("Shape of X_val: ", X_val_m1.shape)
print("Shape of X_test: ", X_test_m1.shape)
print("Shape of Y_train: ", Y_train_m1.shape)
print("Shape of Y_val: ", Y_val_m1.shape)
print("Shape of Y_test: ", Y_test_m1.shape)

#### 1-a: Model development with training and validation datasets

In [ ]:
# Define the tuner function to create a TF binary logistic regression model
def model_tuner(hp):
    tf.keras.backend.clear_session()
    tuner = tf.keras.Sequential()

    # Input layer
    tuner.add(tf.keras.Input(shape=(X_train_m1.shape[1],)))  # Input Dim

    # Hidden layers
    for i in range(hp.Int("num_layers", min_value=1, max_value=3)):
        tuner.add(
            tf.keras.layers.Dense(
                units=hp.Int(f"units_{i}", min_value=16, max_value=128, step=16),
                activation=hp.Choice(
                    f"activation_{i}", values=["relu", "tanh", "sigmoid"]
                ),
            )
        )

    # Output layer
    tuner.add(
        tf.keras.layers.Dense(
            units=1,  # Output dim
            use_bias=True,  # Use a bias (intercept) param
            activation="sigmoid",  # Using sigmoid for binary classification
            kernel_initializer="glorot_uniform",
            bias_initializer="zeros",
        )
    )

    # Learning rate tuning
    learning_rate = hp.Float(
        "learning_rate", min_value=0.0001, max_value=0.5, sampling="log"
    )

    # Optimizer tuning
    optimizer_choice = hp.Choice("optimizer", ["adam", "sgd"])
    if optimizer_choice == "adam":
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_choice == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    # Compile and return the tuner
    tuner.compile(
        optimizer=optimizer,
        loss=hp.Choice("loss", ["binary_crossentropy", "mse"]),
        metrics=["accuracy"],
    )
    return tuner


# Hyperparameter Tuning
tuner = kt.RandomSearch(model_tuner, objective="val_loss", max_trials=10)
tuner.search(
    X_train_m1, Y_train_m1, validation_data=(X_val_m1, Y_val_m1), epochs=50, verbose=2
)
hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"The best hyperparameters are:\n{hyperparameters}")

# Create and fit the model on the training dataset
model_tf = tuner.hypermodel.build(hyperparameters)
history = model_tf.fit(
    X_train_m1,
    Y_train_m1,
    validation_data=(X_val_m1, Y_val_m1),
    epochs=50,
    verbose=False,
)
print(f"\nAccuracy on the training dataset: {history.history['accuracy'][-1]:.2%}")
print(f"Accuracy on the validation dataset: {history.history['val_accuracy'][-1]:.2%}")

# Generate a plot
fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].plot(
    [i + 1 for i in range(len(history.history["loss"]))],
    history.history["loss"],
    label="loss values from \nthe training dataset",
)
ax[0].plot(
    [i + 1 for i in range(len(history.history["val_loss"]))],
    history.history["val_loss"],
    label="loss values from \nthe validation dataset",
)
ax[1].plot(
    [i + 1 for i in range(len(history.history["accuracy"]))],
    history.history["accuracy"],
    label="accuracy values from \nthe training dataset",
    marker=".",
)  # To avoid overlap
ax[1].plot(
    [i + 1 for i in range(len(history.history["val_accuracy"]))],
    history.history["val_accuracy"],
    label="accuracy values from \nthe validation dataset",
)

# Add title, axes, and legend
ax[0].set_title("Loss value for each epoch")
ax[1].set_title("Accuracy value for each epoch")
ax[0].set_xlabel("Epochs")
ax[1].set_xlabel("Epochs")
ax[0].set_ylabel("Loss Value for each epoch")
ax[1].set_ylabel("Accuracy Value for each epoch")
ax[0].legend()
ax[1].legend()
plt.tight_layout()
plt.show()

#### 1-b: Model evaluation with training and testing datasets

In [ ]:
# Calculate the predicted values
Y_train_hat_m1 = (model_tf.predict(X_train_m1) >= 0.5).astype(int)
Y_test_hat_m1 = (model_tf.predict(X_test_m1) >= 0.5).astype(int)

# Calculate accuracy values on both train and test datasets
accuracy_train_m1 = accuracy_score(Y_train_m1, Y_train_hat_m1)
accuracy_test_m1 = accuracy_score(Y_test_m1, Y_test_hat_m1)
print(f"\nThe aggregate accuracy from the training dataset is: {accuracy_train_m1:.2%}")
print(f"The aggregate accuracy from the testing dataset is: {accuracy_test_m1:.2%}")


# Plot the confusion matrix
cm_m1 = confusion_matrix(Y_test_m1, Y_test_hat_m1)
disp = ConfusionMatrixDisplay(cm_m1, display_labels=["No readmission", "Readmitted"])
disp.plot()
plt.grid(False)  # Hide the white grid lines
plt.title("Confusion Matrix on The Test Dataset")
plt.show()

# Evaluation from the confusion matrix
precision_m1 = cm_m1[1][1] / cm_m1[:, 1].sum()
recall_m1 = cm_m1[1][1] / cm_m1[1].sum()
print(f"Precisions: {precision_m1:.2%}")
# print(f'False Positives: {cm_m1[0][1]}')
print(f"Recalls: {recall_m1:.3%}")
print(f"False Negatives: {cm_m1[1][0]}")
print(f"\nF1 score: {2 * precision_m1 * recall_m1 / (precision_m1 + recall_m1):.2%}")

### Model 2 - Random Forest

In [ ]:
# Make copies for model 2 X datasets
X_train_m2 = X_train.copy()
X_val_m2 = X_val.copy()
X_test_m2 = X_test.copy()

# Make copies for model 2 Y datasets
Y_train_m2 = Y_train.copy()
Y_val_m2 = Y_val.copy()
Y_test_m2 = Y_test.copy()

# Reprint the shapes for each dataset
print("Shape of X_train: ", X_train_m2.shape)
print("Shape of X_val: ", X_val_m2.shape)
print("Shape of X_test: ", X_test_m2.shape)
print("Shape of Y_train: ", Y_train_m2.shape)
print("Shape of Y_val: ", Y_val_m2.shape)
print("Shape of Y_test: ", Y_test_m2.shape)

#### 2-a: Model development with training and validation datasets

In [ ]:
# Hyperparameter Tuning
tuning_options_m2 = {
    "n_estimators": [100, 200, 300],  # number of trees
    "max_depth": [6, 8, 12, 16],  # maximum depth of each tree
    "min_samples_split": [2, 5, 10],
}  # Min samples to split for each step

# Create and fit the model on the training dataset
model_rf = RandomizedSearchCV(
    RandomForestClassifier(),  # Estimator - Random Forest
    param_distributions=tuning_options_m2,
    n_iter=30,  # Number of parameter settings sampled
    n_jobs=-1,  # Optimize CPU
    scoring="f1",
)  # Evaluation metric

clf_forest = model_rf.fit(X_train_m2, Y_train_m2)
print(
    f"Accuracy on the training dataset: {clf_forest.score(X_train_m2, Y_train_m2):.2%}"
)
print(f"Accuracy on the validation dataset: {clf_forest.score(X_val_m2, Y_val_m2):.2%}")

# Obtain the top 5 importance feature names and their importance scores
print("\nTop 5 important features:")
print(
    pd.DataFrame(
        {
            "features": X_train_m2.columns,
            "importance": clf_forest.best_estimator_.feature_importances_,
        }
    )
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
    .head()
)

#### 2-b: Model evaluation with training and testing datasets

In [ ]:
# Calculate the predicted values
Y_train_hat_m2 = model_rf.predict(X_train_m2)
Y_test_hat_m2 = model_rf.predict(X_test_m2)

# Calculate accuracy values on both train and test datasets
accuracy_train_m2 = accuracy_score(Y_train_m2, Y_train_hat_m2)
accuracy_test_m2 = accuracy_score(Y_test_m2, Y_test_hat_m2)
print(f"\nThe aggregate accuracy from the training dataset is: {accuracy_train_m2:.2%}")
print(
    f"The aggregate accuracy from the testing dataset is: {accuracy_test_m2:.2%}",
    end="\n\n",
)
print(
    classification_report(
        Y_test_m2, Y_test_hat_m2, target_names=["No readmission", "Readmission"]
    )
)

# Plot the confusion matrix
cm_m2 = confusion_matrix(Y_test_m2, Y_test_hat_m2)
disp = ConfusionMatrixDisplay(cm_m2, display_labels=["No readmission", "Readmitted"])
disp.plot()
plt.grid(False)  # Hide the white grid lines
plt.title("Confusion Matrix on The Test Dataset")
plt.show()

# Evaluation from the confusion matrix
precision_m2 = cm_m2[1][1] / cm_m2[:, 1].sum()
recall_m2 = cm_m2[1][1] / cm_m2[1].sum()
print(f"Precisions: {precision_m2:.2%}")
# print(f'False Positives: {cm_m2[0][1]}')
print(f"Recalls: {recall_m2:.3%}")
print(f"False Negatives: {cm_m2[1][0]}")
print(f"\nF1 score: {2 * precision_m2 * recall_m2 / (precision_m2 + recall_m2):.2%}")

### Model 3 - XGBoost

In [ ]:
# Make copies for model 3 X datasets
X_train_m3 = X_train.copy()
X_val_m3 = X_val.copy()
X_test_m3 = X_test.copy()

# Make copies for model 3 Y datasets
Y_train_m3 = Y_train.copy()
Y_val_m3 = Y_val.copy()
Y_test_m3 = Y_test.copy()

# Reprint the shapes for each dataset
print("Shape of X_train: ", X_train_m3.shape)
print("Shape of X_val: ", X_val_m3.shape)
print("Shape of X_test: ", X_test_m3.shape)
print("Shape of Y_train: ", Y_train_m3.shape)
print("Shape of Y_val: ", Y_val_m3.shape)
print("Shape of Y_test: ", Y_test_m3.shape)

#### 3-a: Model development with training and validation datasets

In [ ]:
# Hyperparameter Tuning
tuning_options_m3 = {
    "n_estimators": [30, 50, 100, 200, 300, 500],  # number of boosting rounds
    "max_depth": [3, 6, 8, 12, 16],  # maximum depth of each tree
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.4],  # step size shrinkage
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 0.1, 0.2, 0.3],
    "min_child_weight": [1, 3, 5, 7],
}

# Create and fit the model on the training dataset
model_xgb = RandomizedSearchCV(
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        use_label_encoder=False,
        scale_pos_weight=len(Y_train[Y_train == 0]) / len(Y_train[Y_train == 1]),
    ),  # Estimator - XGBoost
    param_distributions=tuning_options_m3,
    n_iter=30,  # Number of parameter settings sampled
    n_jobs=-1,  # Optimize CPU
    scoring="f1",
)  # Evaluation metric


clf_xgb = model_xgb.fit(X_train_m3, Y_train_m3)
print(f"Accuracy on the training dataset: {clf_xgb.score(X_train_m3, Y_train_m3):.2%}")
print(f"Accuracy on the validation dataset: {clf_xgb.score(X_val_m3, Y_val_m3):.2%}")

# Obtain the top 5 importance feature names and their importance scores
print("\nTop 5 important features:")
print(
    pd.DataFrame(
        {
            "features": X_train_m3.columns,
            "importance": clf_xgb.best_estimator_.feature_importances_,
        }
    )
    .sort_values(by="importance", ascending=False)
    .reset_index(drop=True)
    .head()
)

#### 3-b: Model evaluation with training and testing datasets

In [ ]:
# Calculate the predicted values
Y_train_hat_m3 = model_xgb.predict(X_train_m3)
Y_test_hat_m3 = model_xgb.predict(X_test_m3)

# Calculate accuracy values on both train and test datasets
accuracy_train_m3 = accuracy_score(Y_train_m3, Y_train_hat_m3)
accuracy_test_m3 = accuracy_score(Y_test_m3, Y_test_hat_m3)
print(f"\nThe aggregate accuracy from the training dataset is: {accuracy_train_m3:.2%}")
print(
    f"The aggregate accuracy from the testing dataset is: {accuracy_test_m3:.2%}",
    end="\n\n",
)
print(
    classification_report(
        Y_test_m3, Y_test_hat_m3, target_names=["No readmission", "Readmission"]
    )
)

# Plot the confusion matrix
cm_m3 = confusion_matrix(Y_test_m3, Y_test_hat_m3)
disp = ConfusionMatrixDisplay(cm_m3, display_labels=["No readmission", "Readmitted"])
disp.plot()
plt.grid(False)  # Hide the white grid lines
plt.title("Confusion Matrix on The Test Dataset")
plt.show()

# Evaluation from the confusion matrix
precision_m3 = cm_m3[1][1] / cm_m3[:, 1].sum()
recall_m3 = cm_m3[1][1] / cm_m3[1].sum()
print(f"Precisions: {precision_m3:.2%}")
# print(f'False Positives: {cm_m3[0][1]}')
print(f"Recalls: {recall_m3:.3%}")
print(f"False Negatives: {cm_m3[1][0]}")
print(f"\nF1 score: {2 * precision_m3 * recall_m3 / (precision_m3 + recall_m3):.2%}")